# 27. ZeRO Optimizer Sim | ZeRO 优化器模拟

**难度：** Hard | **环境：** CPU-first | **标签：** `并行通信`, `ZeRO`, `参数切分` | **目标人群：** 并行通信学习者

---

## 本节导读

数据并行能把 batch 分到多张卡上计算，但它有一个明显浪费：每张卡都保存完整参数、完整梯度和完整优化器状态。模型一大，真正压垮显存的往往不是前向本身，而是这些训练状态在每张卡上的重复存储。

ZeRO 的思路是把这些重复状态拆开，让不同 GPU 只维护自己负责的一部分。本节用一个简化模拟先看 ZeRO-1：优化器状态如何切分，Reduce-Scatter 和 All-Gather 分别解决什么问题。完成后，你应该能理解 ZeRO 为什么能降低单卡显存占用，也能把它和后面的 Pipeline / Tensor Parallelism 区分开。

**关键词：** `ZeRO`, `Reduce-Scatter`, `All-Gather`

---


## 前置阅读

**导语：** 进入本节前，先能区分参数、梯度和优化器状态，并能从显存账本中看出它们为什么会在每张 GPU 上重复保存。

- [P0: 11. PyTorch Optimizers and Loss | PyTorch 优化器与损失函数](../00_Prerequisites/11_PyTorch_Optimizers_and_Loss.ipynb)
- [P0: 13. Simple Neural Network Training | 简单神经网络训练循环](../00_Prerequisites/13_Simple_Neural_Network_Training.ipynb)
- [P0: 20. Profiling and Memory Ledger | 性能剖析与显存账本](../00_Prerequisites/20_Profiling_and_Memory_Ledger.ipynb)


---

### Step 1：从重复训练状态理解 ZeRO

> **传统的 Data Parallel (DP，数据并行)：**

> 每张卡都有一份完整的模型权重、完整的梯度、完整的优化器状态。
> 各个卡算完自己这批数据的梯度后，进行 `All-Reduce` 求平均。然后每张卡用自己的优化器更新自己完整的权重。
> **痛点：严重浪费！每张卡都在重复保存一样的优化器状态和重复做一样的参数更新。**

> **ZeRO-1 的机制：**
> 1. 每张卡依然有完整的模型权重和完整的梯度（前向和反向传播与 DP 完全一样）。
> 2. **切分：** 优化器状态被切分成 $N$ 份（假设有 $N$ 张卡），每张卡只负责维护 $\frac{1}{N}$ 的优化器状态，并只负责更新这 $\frac{1}{N}$ 的模型权重。
> 3. **通信：** 反向传播结束后，不需要对所有梯度做 `All-Reduce`，而是做 `Reduce-Scatter`，让每张卡只拿到属于自己那 $\frac{1}{N}$ 权重的平均梯度。
> 4. 每张卡更新自己负责的 $\frac{1}{N}$ 权重后，再通过 `All-Gather` 将更新后的片段广播给所有卡，拼合出完整的新权重。

![ZeRO：把训练状态从每卡复制改成分片](../docs/public/02_PyTorch_Algorithms/27_zero_sharding.svg)


### Step 2：理解 ZeRO-1 的状态与通信流

ZeRO-1 的变化可以沿一次参数更新观察：

1. 反向传播结束后，每张 GPU 暂时都有完整梯度；
2. Reduce-Scatter 对梯度求和并分发，每张 GPU 只保留自己负责参数片段的平均梯度；
3. 每张 GPU 用本地优化器状态更新自己的参数片段；
4. All-Gather 收集更新后的参数片段，使下一轮前向仍能看到完整参数。

这里的核心交换是“少保存优化器状态，增加一次状态分发与参数同步”。参数和梯度仍然保持完整，因此 ZeRO-1 与 ZeRO-2、ZeRO-3 的差异要到下一步的分片范围比较中理解。


### Step 3：比较 ZeRO-1、ZeRO-2 与 ZeRO-3

Step 1 说明了重复状态为什么需要分片，Step 2 展示了 ZeRO-1 如何完成一次局部更新。下面用同一份 FP16 + Adam 账本比较三种策略，选择时同时看单卡状态、通信和模型规模。

以模型参数量 $\Phi$ 为单位，参数占 $2\Phi$ bytes、梯度占 $2\Phi$ bytes、Adam 优化器状态占 $12\Phi$ bytes。此处把 ZeRO-1/2/3 的状态账本和通信代价放在同一张表中，避免与解析区重复。

| 策略 | 分片对象 | 单卡状态（近似） | 主要通信 | 适用信号 |
|---|---|---:|---|---|
| Data Parallel | 无分片 | $16\Phi$ | All-Reduce 梯度 | 模型和训练状态都能放下 |
| ZeRO-1 | 优化器状态 | $2\Phi + 2\Phi + 12\Phi/N$ | Reduce-Scatter + All-Gather | 优化器状态主导显存 |
| ZeRO-2 | 优化器状态 + 梯度 | $2\Phi + 14\Phi/N$ | 梯度 Reduce-Scatter | 梯度也成为压力 |
| ZeRO-3 | 优化器状态 + 梯度 + 参数 | $16\Phi/N$ | 按层 All-Gather，通信更频繁 | 参数本身已经放不下 |

当 $N=8$ 时，ZeRO-1 只把优化器状态部分降为 $1/8$；ZeRO-3 的单卡状态最低，但更依赖高速互联和稳定的参数聚合。

![ZeRO 分阶段：省下的状态对应新增的通信](../docs/public/02_PyTorch_Algorithms/27_zero_stages.svg)


### Step 4：CPU 实现——验证 ZeRO-1 的分片与局部更新

下面用固定 2 GPU 的逻辑分片模拟一次 ZeRO-1 更新。题目区只实现参数归属、局部优化器状态和局部参数更新；测试区分别验证状态分片、更新结果与完整参数可见性。

| 函数或对象 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|---|---|---|---|
| ZeRO1_Optimizer_Sim.__init__ | 参数分片与局部优化器状态 | 每个参数只归属于一个 GPU；状态只为本地参数创建 | 状态字典大小、参数分片覆盖 |
| ZeRO1_Optimizer_Sim.step | 使用 Reduce-Scatter 后的局部梯度更新参数 | 只更新当前 GPU 负责的参数，并写回对应状态 | 局部更新值、参数可见性 |
| zero_grad | 清理已有梯度 | 不改变参数归属和优化器状态 | 与完整参数列表兼容 |


In [ ]:
import torch
import torch.nn as nn

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # 为了演示切分，我们用一个包含偶数个参数的线性层
        self.fc1 = nn.Linear(dim, dim, bias=False)
        self.fc2 = nn.Linear(dim, dim, bias=False)
        
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

class ZeRO1_Optimizer_Sim:
    """
    模拟 2 张 GPU 上的 ZeRO-1 优化器行为。
    为了简化，我们假设模型的所有参数都被展平 (Flatten) 为一个一维张量，然后均分为 2 份。
    """
    def __init__(self, model_params, lr=0.1, num_gpus=2):
        self.lr = lr
        self.num_gpus = num_gpus
        
        # 将所有参数的引用收集起来
        self.params = list(model_params)
        
        # ==========================================
        # TODO 1：完成参数分片，并记录每个 rank 的参数集合
        # 变量提示（每个变量各占一行）：
        # half_idx = len(self.params) // self.num_gpus
        # self.gpu_partitions = {gpu_id: ...}
        # ==========================================
        pass

        # ==========================================
        # TODO 2：为每个 rank 初始化与本地参数对应的优化器状态
        # 变量提示（每个变量各占一行）：
        # gpu_id = ...
        # self.optimizer_states = {gpu_id: {id(p): torch.zeros_like(p.data) for p in params}}
        # ==========================================
        pass

        
    def step(self, gradients_from_all_gpus: dict):
        """
        gradients_from_all_gpus 模拟了 Reduce-Scatter 的结果。
        结构：{gpu_id: [它负责的参数的平均梯度]}
        """
        # ==========================================
        # TODO 3：完成一个 rank 的局部状态更新
        # 变量提示（每个变量各占一行）：
        # params = self.gpu_partitions[gpu_id]
        # grads = gradients_from_all_gpus[gpu_id]
        # states = self.optimizer_states[gpu_id]
        # 对每个 (p, g)，读取 states[id(p)]，更新状态并写回 p.data。
        # ==========================================
        for gpu_id in range(self.num_gpus):
            pass
            
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


In [ ]:
# 测试你的实现
def test_zero1_sim():
    """验证 ZeRO-1 的状态分片、局部更新和参数可见性。"""
    try:
        torch.manual_seed(42)
        model = SimpleModel(dim=4)
        optimizer = ZeRO1_Optimizer_Sim(model.parameters(), lr=0.1, num_gpus=2)
        
        # 保存初始权重用于对比
        initial_w1 = model.fc1.weight.data.clone()
        initial_w2 = model.fc2.weight.data.clone()
        
        # 模拟反向传播产生的平均梯度 (Reduce-Scatter 的结果)
        # 假设 fc1 是 GPU 0 负责，fc2 是 GPU 1 负责
        simulated_reduce_scatter_grads = {
            0: [torch.ones_like(model.fc1.weight)],  # GPU 0 收到 fc1 的梯度
            1: [torch.full_like(model.fc2.weight, 2.0)] # GPU 1 收到 fc2 的梯度
        }
        
        # 验证优化器状态切分 (ZeRO-1 的核心显存节约)
        assert len(optimizer.optimizer_states[0]) == 1, "GPU 0 应该只维护 fc1 的状态"
        assert len(optimizer.optimizer_states[1]) == 1, "GPU 1 应该只维护 fc2 的状态"
        
        # 执行更新
        optimizer.step(simulated_reduce_scatter_grads)
        
        # 验证更新结果是否正确应用到了原模型上 (隐式的 All-Gather)
        diff_w1 = initial_w1 - model.fc1.weight.data
        diff_w2 = initial_w2 - model.fc2.weight.data
        
        # 预期：momentum 从 0 变成 1，w1 减去 lr * 1 = 0.1
        # 预期：momentum 从 0 变成 2，w2 减去 lr * 2 = 0.2
        assert torch.allclose(diff_w1, torch.full_like(diff_w1, 0.1)), "GPU 0 负责的权重更新错误！"
        assert torch.allclose(diff_w2, torch.full_like(diff_w2, 0.2)), "GPU 1 负责的权重更新错误！"
        
        print("✅ ZeRO-1 优化器状态切分与更新逻辑测试通过！")
        
    except NotImplementedError:
        print("请先完成 TODO 代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError, AssertionError, RuntimeError) as e:
        if isinstance(e, AttributeError):
            print("代码未完成，无法找到必要的属性")
        elif isinstance(e, NameError):
            print("代码可能未完成，导致了变量未定义")
        elif isinstance(e, TypeError):
            print("代码可能未完成，导致了类型错误")
        elif isinstance(e, ValueError):
            print("代码可能未完成，导致了张量维度错误")
        elif isinstance(e, AssertionError):
            print("代码可能未完成，导致了断言失败")
        else:
            print("代码可能未完成，导致了运行时错误")
        raise NotImplementedError("请先完成 TODO 代码！") from e
    except Exception as e:
        print(f"❌ 测试失败: {e}")
        raise

def test_zero1_partition_state():
    # 机制：每个 rank 只持有自己负责参数的优化器状态。
    model = SimpleModel(dim=4)
    optimizer = ZeRO1_Optimizer_Sim(model.parameters(), num_gpus=2)
    assert all(len(states) == 1 for states in optimizer.optimizer_states.values())

def test_zero1_local_update_value():
    # 机制：Reduce-Scatter 后只更新本 rank 的参数片段。
    model = SimpleModel(dim=4)
    optimizer = ZeRO1_Optimizer_Sim(model.parameters(), lr=0.1, num_gpus=2)
    before = [p.detach().clone() for p in model.parameters()]
    grads = {0: [torch.ones_like(model.fc1.weight)], 1: [torch.ones_like(model.fc2.weight)]}
    optimizer.step(grads)
    assert all(not torch.equal(a, b) for a, b in zip(before, model.parameters()))

def test_zero1_parameter_visibility():
    # 机制：局部更新后，完整模型参数仍可被统一读取。
    model = SimpleModel(dim=4)
    optimizer = ZeRO1_Optimizer_Sim(model.parameters(), num_gpus=2)
    assert sum(len(v) for v in optimizer.gpu_partitions.values()) == len(list(model.parameters()))

test_zero1_partition_state()
test_zero1_local_update_value()
test_zero1_parameter_visibility()
test_zero1_sim()

---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---

## 参考代码与解析

### 代码

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim, bias=False)
        self.fc2 = nn.Linear(dim, dim, bias=False)
        
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

class ZeRO1_Optimizer_Sim:
    """
    模拟 2 张 GPU 上的 ZeRO-1 优化器行为。
    """
    def __init__(self, model_params, lr=0.1, num_gpus=2):
        self.lr = lr
        self.num_gpus = num_gpus
        
        # 将所有参数的引用收集起来
        self.params = list(model_params)
        
        # TODO 1: 将参数切分给不同的 GPU 负责
        half_idx = len(self.params) // 2
        self.gpu_partitions = {
            0: self.params[:half_idx],
            1: self.params[half_idx:]
        }
        
        # TODO 2: 为每个 GPU 初始化局部状态
        self.optimizer_states = {
            0: {id(p): torch.zeros_like(p.data) for p in self.gpu_partitions[0]},
            1: {id(p): torch.zeros_like(p.data) for p in self.gpu_partitions[1]}
        }
        
    def step(self, gradients_from_all_gpus: dict):
        """
        gradients_from_all_gpus 模拟了 Reduce-Scatter 的结果。
        """
        # TODO 3: 模拟每张卡只更新自己负责的那部分权重
        for gpu_id in range(self.num_gpus):
            params = self.gpu_partitions[gpu_id]
            grads = gradients_from_all_gpus[gpu_id]
            states = self.optimizer_states[gpu_id]
            
            for p, g in zip(params, grads):
                # 更新动量
                momentum = states[id(p)]
                momentum = momentum + g  # 简化版：直接累加梯度作为动量
                states[id(p)] = momentum
                
                # 使用动量更新参数
                p.data = p.data - self.lr * momentum
                
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


### 解析

**1. TODO 1：参数分片**
- half_idx 将两个参数对象分到两个 GPU；每个参数只出现在一个分片中。
- 这是本节的教学简化，真实 ZeRO 会按参数量或 bucket 做更细粒度的分片。

**2. TODO 2：初始化局部优化器状态**
- 每个 GPU 只为自己负责的参数创建状态张量。
- 参数对象的 id 只用于把状态映射回参数对象，不代表真实分布式 rank 的通信标识。

**3. TODO 3：局部状态更新**
- gradients_from_all_gpus 被视为 Reduce-Scatter 已经完成后的局部梯度。
- 每个 GPU 更新自己的参数和动量；All-Gather 的完整同步效果由共享参数引用的测试观察。

Step 3 的策略表已经承担 ZeRO-1/2/3 的对比，因此解析区不再重复一张相同表格。


### Step 5（可选）：GPU ZeRO 分片 benchmark

CPU 模拟用于验证 ZeRO-1 的分片不变量；下面使用 DeepSpeed 在真实多卡环境中比较 DDP、ZeRO-1 以及可选的 ZeRO-2/3。CPU 结果不能替代真实多卡证据。


#### 5.1 环境与固定 workload

至少使用两张 GPU，固定模型、dtype、global batch、micro-batch、序列长度、warmup 与 repeats。G0 为 DDP baseline，G1 为 ZeRO-1。

In [ ]:
# 5.1：默认关闭；先固定 G0/G1 的模型、数据与训练口径。
RUN_GPU_EXPERIMENT = False
WORLD_SIZE = 2
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DTYPE = 'bf16'
GLOBAL_BATCH_SIZE = 4
MICRO_BATCH_SIZE = 1
SEQ_LEN = 512
WARMUP = 2
REPEATS = 5
RESULT_PATH = 'benchmarks/results/27_zero_benchmark.json'
# 例如：['deepspeed', '--num_gpus', '2', 'train_zero.py', '--zero-stage', '1']
BENCHMARK_COMMAND = None
print({'run': RUN_GPU_EXPERIMENT, 'world_size': WORLD_SIZE, 'model': MODEL_ID, 'dtype': DTYPE, 'result': RESULT_PATH})


#### 5.2 配置与执行

使用 DeepSpeed 启动真实训练；默认关闭，框架不可用、GPU 数不足、OOM 或 NCCL 失败时记录 failure。

In [ ]:
# 5.2：执行真实 DeepSpeed 训练；命令必须将 G0/G1 结果写入 RESULT_PATH。
if RUN_GPU_EXPERIMENT:
    if not BENCHMARK_COMMAND:
        raise ValueError('请先填写 DeepSpeed BENCHMARK_COMMAND；CPU 模拟不能替代真实 ZeRO benchmark。')
    import subprocess
    subprocess.run(BENCHMARK_COMMAND, check=True)
else:
    print('GPU benchmark 默认关闭。')


#### 5.3 读取结果、解释指标与形成决策

读取真实运行生成的 JSON，确认 workload、hardware、metrics、evidence_level、failure 和 decision 齐全。先确认单卡状态显存是否下降，再判断通信代价是否吞掉训练吞吐；CPU 分片正确性不能替代真实训练证据。

In [ ]:
# 5.3：读取真实结果；没有 JSON 时不输出性能结论。
import json
from pathlib import Path
if Path(RESULT_PATH).exists():
    result = json.loads(Path(RESULT_PATH).read_text())
    required = {'strategy', 'workload', 'hardware', 'metrics', 'evidence_level', 'failure', 'decision'}
    missing = required - set(result)
    if missing:
        raise ValueError(f'结果 JSON 缺少字段：{sorted(missing)}')
    print(result)
else:
    print(f'尚无真实 ZeRO 结果：{RESULT_PATH}')


## 相关阅读

ZeRO 的核心是训练状态分片；读完本节后，可以再用论文、DeepSpeed 实现和并行策略页面核对通信与显存取舍。

- [ZeRO 原论文：Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054)
- [DeepSpeed ZeRO 官方文档](https://www.deepspeed.ai/tutorials/zero/)
- [P1: 通信拓扑与分布式基石](../01_Hardware_Math_and_Systems/05_Communication_Topologies.ipynb)
- [28. Pipeline 并行微批次](../02_PyTorch_Algorithms/28_Pipeline_Parallelism_MicroBatch.ipynb)
- [29. Tensor 并行模拟](../02_PyTorch_Algorithms/29_Tensor_Parallelism_Sim.ipynb)
- [P1: 并行策略决策框架](../01_Hardware_Math_and_Systems/26_Parallel_Strategy_Decision_Framework.ipynb)
